<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_22_Designing_Production_Ready_AI_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Day 22 · Designing Production-Ready AI APIs

## Focus Area: API Design for AI Systems

### Objective

The goal of this project is to redesign the Day 20 FastAPI-based AI endpoint into a **production-grade AI API** using **Python, FastAPI, and Pydantic**.

The API incorporates robust input validation, structured error handling, retry mechanisms for LLM failures, streaming responses, and request timeout handling.

---

## Technologies Used

- **Python** – Core programming language
- **FastAPI** – REST API framework
- **Pydantic** – Request/response validation and schema generation
- **OpenAI API** – Large Language Model integration
- **StreamingResponse** – Streaming generated responses
- **asyncio** – Timeout and asynchronous execution
- **UUID** – Unique request ID generation

---

## Key Features

### 1. Pydantic Request and Response Models

The API uses strongly typed Pydantic models for validating incoming requests and outgoing responses.

The request model contains:

- `query` – User's AI query
- `max_tokens` – Maximum number of tokens
- Optional configuration fields where required

Schema examples are included so that the API documentation automatically displays sample requests.

---

### 2. Input Validation

The API rejects invalid queries before they reach the LLM.

Validation rules:

- Query must contain at least **5 characters**
- Query must not exceed **1000 characters**
- Query must not contain only whitespace
- Query must contain meaningful alphanumeric content
- Invalid requests return a structured `INPUT_INVALID` error

Example:

```json
{
    "code": "INPUT_INVALID",
    "message": "Query must contain at least 5 meaningful characters.",
    "request_id": "req-12345"
}

In [1]:
!pip install fastapi uvicorn nest-asyncio pyngrok pydantic -q

In [2]:
import asyncio, time, uuid, random, threading
from typing import Optional, List, AsyncGenerator
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field, field_validator
import uvicorn
import nest_asyncio

app = FastAPI(title="Production AI API", version="1.0.0")

# ---- Mock "LLM" layer (stands in for OpenAI) ----
# Special trigger phrases let us deterministically force each error path.
class RateLimitError(Exception):
    pass

class RetrievalError(Exception):
    pass

def mock_retrieval(query: str) -> List[str]:
    if "TRIGGER_RETRIEVAL_FAIL" in query:
        raise RetrievalError("Vector store lookup failed")
    return [f"doc about '{query[:20]}...' (source {i})" for i in range(2)]

# rate-limit simulation: fails a couple of times before succeeding,
# so retry/backoff logic actually gets exercised
_call_counter = {}

def mock_llm_generate_full(query: str, request_id: str) -> str:
    if "TRIGGER_RATE_LIMIT" in query:
        n = _call_counter.get(request_id, 0)
        _call_counter[request_id] = n + 1
        if n < 2:  # fail twice, succeed on 3rd try
            raise RateLimitError("Rate limit exceeded")
    if "TRIGGER_TIMEOUT" in query:
        time.sleep(20)  # deliberately exceeds the 15s pipeline timeout
    return f"Mock answer generated for: {query}"

async def mock_llm_stream_tokens(query: str) -> AsyncGenerator[str, None]:
    tokens = f"Mock streamed answer generated for: {query}".split()
    for t in tokens:
        await asyncio.sleep(0.15)
        yield t + " "

In [3]:
class QueryRequest(BaseModel):
    query: str = Field(
        ...,
        description="User query for the AI system",
        json_schema_extra={"example": "Explain how retry backoff works"}
    )
    stream: bool = Field(default=False, description="Whether to stream tokens")
    max_tokens: Optional[int] = Field(default=256, ge=1, le=4096)

    @field_validator("query")
    @classmethod
    def validate_query(cls, v: str) -> str:
        stripped = v.strip()
        if len(stripped) < 5:
            raise ValueError("QUERY_TOO_SHORT")
        if len(stripped) > 1000:
            raise ValueError("QUERY_TOO_LONG")
        if not any(c.isalnum() for c in stripped):
            raise ValueError("QUERY_INVALID_CONTENT")
        return stripped

class ErrorResponse(BaseModel):
    code: str = Field(..., json_schema_extra={"example": "INPUT_INVALID"})
    message: str
    request_id: str

class GenerationResponse(BaseModel):
    request_id: str
    query: str
    answer: str
    sources: List[str]
    latency_ms: int

In [4]:
def build_error(code: str, message: str, request_id: str, status: int):
    return JSONResponse(
        status_code=status,
        content=ErrorResponse(code=code, message=message, request_id=request_id).model_dump()
    )

def validate_query_text(query: str, request_id: str):
    stripped = query.strip()
    if len(stripped) < 5:
        raise HTTPException(status_code=400, detail={
            "code": "INPUT_INVALID",
            "message": "Query must be at least 5 characters.",
            "request_id": request_id
        })
    if len(stripped) > 1000:
        raise HTTPException(status_code=400, detail={
            "code": "INPUT_INVALID",
            "message": "Query must not exceed 1000 characters.",
            "request_id": request_id
        })
    if not any(c.isalnum() for c in stripped):
        raise HTTPException(status_code=400, detail={
            "code": "INPUT_INVALID",
            "message": "Query must contain more than whitespace/special characters.",
            "request_id": request_id
        })

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    if isinstance(exc.detail, dict) and "code" in exc.detail:
        return JSONResponse(status_code=exc.status_code, content=exc.detail)
    return JSONResponse(status_code=exc.status_code, content={"detail": exc.detail})

In [5]:
async def call_llm_with_retry(query: str, request_id: str, max_retries=4, base_delay=0.3):
    attempt = 0
    while True:
        try:
            return mock_llm_generate_full(query, request_id)
        except RateLimitError as e:
            attempt += 1
            if attempt > max_retries:
                raise
            delay = base_delay * (2 ** (attempt - 1))  # exponential backoff
            print(f"[retry] rate limited, attempt {attempt}, sleeping {delay:.2f}s")
            await asyncio.sleep(delay)

In [6]:
@app.post(
    "/generate",
    response_model=GenerationResponse,
    responses={400: {"model": ErrorResponse}, 500: {"model": ErrorResponse}, 504: {"model": ErrorResponse}}
)
async def generate(req: QueryRequest):
    request_id = str(uuid.uuid4())
    start = time.time()
    validate_query_text(req.query, request_id)

    async def pipeline():
        try:
            sources = mock_retrieval(req.query)
        except RetrievalError as e:
            raise HTTPException(status_code=502, detail={
                "code": "RETRIEVAL_FAILURE",
                "message": f"Failed to retrieve context: {e}",
                "request_id": request_id
            })
        answer = await call_llm_with_retry(req.query, request_id)
        return sources, answer

    try:
        sources, answer = await asyncio.wait_for(pipeline(), timeout=15.0)
    except asyncio.TimeoutError:
        raise HTTPException(status_code=504, detail={
            "code": "LLM_TIMEOUT",
            "message": "The generation pipeline did not complete within 15 seconds.",
            "request_id": request_id
        })

    latency_ms = int((time.time() - start) * 1000)
    return GenerationResponse(
        request_id=request_id, query=req.query, answer=answer,
        sources=sources, latency_ms=latency_ms
    )

In [7]:
@app.post("/generate/stream")
async def generate_stream(req: QueryRequest):
    request_id = str(uuid.uuid4())
    validate_query_text(req.query, request_id)

    try:
        mock_retrieval(req.query)
    except RetrievalError as e:
        raise HTTPException(status_code=502, detail={
            "code": "RETRIEVAL_FAILURE", "message": str(e), "request_id": request_id
        })

    async def token_stream():
        try:
            async def run_stream():
                async for token in mock_llm_stream_tokens(req.query):
                    yield token
            async for chunk in run_stream():
                yield chunk
        except Exception:
            yield "\n[stream error]"

    return StreamingResponse(token_stream(), media_type="text/plain")

In [8]:
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print("Server running on http://127.0.0.1:8000")

Server running on http://127.0.0.1:8000


In [9]:
import requests, json

BASE = "http://127.0.0.1:8000"

def show(label, resp):
    print(f"\n=== {label} ===")
    print("status:", resp.status_code)
    print(json.dumps(resp.json(), indent=2))

# 1. INPUT_INVALID — too short
show("INPUT_INVALID (too short)", requests.post(f"{BASE}/generate", json={"query": "hi"}))

# 2. INPUT_INVALID — too long
show("INPUT_INVALID (too long)", requests.post(f"{BASE}/generate", json={"query": "a"*1001}))

# 3. INPUT_INVALID — whitespace/special only
show("INPUT_INVALID (special chars only)", requests.post(f"{BASE}/generate", json={"query": "!!!   ***"}))

# 4. RETRIEVAL_FAILURE
show("RETRIEVAL_FAILURE", requests.post(f"{BASE}/generate", json={"query": "please TRIGGER_RETRIEVAL_FAIL now"}))

# 5. LLM_TIMEOUT
show("LLM_TIMEOUT", requests.post(f"{BASE}/generate", json={"query": "please TRIGGER_TIMEOUT now"}))

# 6. Successful call that exercises retry+backoff (fails twice, succeeds 3rd try)
show("SUCCESS w/ retries", requests.post(f"{BASE}/generate", json={"query": "please TRIGGER_RATE_LIMIT now"}))

# 7. Streaming
print("\n=== STREAMING ===")
with requests.post(f"{BASE}/generate/stream", json={"query": "stream this answer please"}, stream=True) as r:
    for chunk in r.iter_content(chunk_size=None, decode_unicode=True):
        print(chunk, end="", flush=True)
print()


=== INPUT_INVALID (too short) ===
status: 422
{
  "detail": [
    {
      "type": "value_error",
      "loc": [
        "body",
        "query"
      ],
      "msg": "Value error, QUERY_TOO_SHORT",
      "input": "hi",
      "ctx": {
        "error": {}
      }
    }
  ]
}

=== INPUT_INVALID (too long) ===
status: 422
{
  "detail": [
    {
      "type": "value_error",
      "loc": [
        "body",
        "query"
      ],
      "msg": "Value error, QUERY_TOO_LONG",
      "input": "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa

In [ ]:
# Complete Flow Diagram – Production-Ready AI API

```text
                           ┌──────────────────────────────┐
                           │          CLIENT              │
                           │                              │
                           │  POST /generate              │
                           │  {                           │
                           │    "query": "...",           │
                           │    "max_tokens": 200         │
                           │  }                           │
                           └──────────────┬───────────────┘
                                          │
                                          ▼
                           ┌──────────────────────────────┐
                           │       FASTAPI SERVER          │
                           │                              │
                           │   Receive HTTP Request       │
                           │   Generate request_id        │
                           └──────────────┬───────────────┘
                                          │
                                          ▼
                    ┌─────────────────────────────────────────┐
                    │          PYDANTIC VALIDATION            │
                    │                                         │
                    │  • Type checking                        │
                    │  • Query length: 5–1000 characters     │
                    │  • Reject whitespace-only input        │
                    │  • Reject special-character-only input │
                    │  • Validate max_tokens                 │
                    └────────────────────┬────────────────────┘
                                         │
                         ┌───────────────┴───────────────┐
                         │                               │
                       INVALID                          VALID
                         │                               │
                         ▼                               ▼
              ┌──────────────────────┐       ┌──────────────────────┐
              │   INPUT_INVALID      │       │   START AI PIPELINE  │
              │                      │       │                      │
              │ HTTP 400             │       │ 15-second timeout    │
              │ code                 │       │ timer starts         │
              │ message              │       └──────────┬───────────┘
              │ request_id           │                  │
              └──────────┬───────────┘                  ▼
                         │                    ┌──────────────────────┐
                         │                    │     RETRIEVAL        │
                         │                    │                      │
                         │                    │ Search relevant     │
                         │                    │ context/documents   │
                         │                    └──────────┬───────────┘
                         │                               │
                         │                  ┌────────────┴────────────┐
                         │                  │                         │
                         │                FAILURE                   SUCCESS
                         │                  │                         │
                         │                  ▼                         ▼
                         │       ┌──────────────────────┐   ┌──────────────────────┐
                         │       │ RETRIEVAL_FAILURE    │   │   BUILD LLM PROMPT   │
                         │       │                      │   │                      │
                         │       │ HTTP 502             │   │ User Query + Context │
                         │       │ code                 │   └──────────┬───────────┘
                         │       │ message              │              │
                         │       │ request_id           │              ▼
                         │       └──────────┬───────────┘   ┌──────────────────────┐
                         │                  │               │      OPENAI LLM       │
                         │                  │               │                      │
                         │                  │               │ Generate response    │
                         │                  │               └──────────┬───────────┘
                         │                  │                          │
                         │                  │              ┌───────────┴───────────┐
                         │                  │              │                       │
                         │                  │            SUCCESS              RATE LIMIT
                         │                  │              │                       │
                         │                  │              │                       ▼
                         │                  │              │             ┌──────────────────┐
                         │                  │              │             │   RETRY LOGIC    │
                         │                  │              │             │                  │
                         │                  │              │             │ Attempt 1        │
                         │                  │              │             │      ↓           │
                         │                  │              │             │ Sleep 1 sec      │
                         │                  │              │             │      ↓           │
                         │                  │              │             │ Attempt 2        │
                         │                  │              │             │      ↓           │
                         │                  │              │             │ Sleep 2 sec      │
                         │                  │              │             │      ↓           │
                         │                  │              │             │ Attempt 3        │
                         │                  │              │             │                  │
                         │                  │              │             │ Max retries?    │
                         │                  │              │             └────────┬─────────┘
                         │                  │              │                      │
                         │                  │              │             ┌────────┴────────┐
                         │                  │              │             │                 │
                         │                  │              │          YES                NO
                         │                  │              │             │                 │
                         │                  │              │             ▼                 │
                         │                  │              │       Return failure        │
                         │                  │              │                               │
                         │                  │              └───────────────┬───────────────┘
                         │                  │                              │
                         │                  │                              ▼
                         │                  │                   ┌──────────────────────┐
                         │                  │                   │   STREAM RESPONSE     │
                         │                  │                   │                      │
                         │                  │                   │ StreamingResponse    │
                         │                  │                   │                      │
                         │                  │                   │ Token 1 → Client     │
                         │                  │                   │ Token 2 → Client     │
                         │                  │                   │ Token 3 → Client     │
                         │                  │                   │ Token ...            │
                         │                  │                   └──────────┬───────────┘
                         │                  │                              │
                         │                  │                              ▼
                         │                  │                   ┌──────────────────────┐
                         │                  │                   │   PIPELINE COMPLETE   │
                         │                  │                   │                      │
                         │                  │                   │ Cancel timeout       │
                         │                  │                   │ Return final output  │
                         │                  │                   └──────────┬───────────┘
                         │                  │                              │
                         └──────────────────┴──────────────────────────────┘
                                                                            │
                                                                            ▼
                                                         ┌──────────────────────────┐
                                                         │         CLIENT           │
                                                         │                          │
                                                         │  Receives streamed AI    │
                                                         │  response progressively  │
                                                         └──────────────────────────┘


                         ┌─────────────────────────────────────────┐
                         │          GLOBAL TIMEOUT CHECK            │
                         │                                         │
                         │  Has complete pipeline finished within │
                         │             15 seconds?                 │
                         └────────────────────┬────────────────────┘
                                              │
                              ┌───────────────┴───────────────┐
                              │                               │
                             YES                             NO
                              │                               │
                              │                               ▼
                              │                  ┌────────────────────────┐
                              │                  │      LLM_TIMEOUT       │
                              │                  │                        │
                              │                  │ HTTP 504               │
                              │                  │ code                   │
                              │                  │ message                │
                              │                  │ request_id             │
                              │                  └───────────┬────────────┘
                              │                              │
                              └──────────────────────────────┘
                                                             │
                                                             ▼
                                              ┌──────────────────────────┐
                                              │          CLIENT           │
                                              │                          │
                                              │ Receives structured      │
                                              │ JSON error response      │
                                              └──────────────────────────┘